# Demo: Deploying a Customized LLM with SageMaker

This notebook demonstrates fine-tuning **Mistral 7B** on SageMaker using JumpStart.

The model is fine-tuned on a customer support dataset so it responds in a consistent, professional, and empathetic tone — behaviour the base model does not exhibit by default.

This notebook walks through:
- Setting up the environment
- Uploading the training dataset
- Fine-tuning Mistral 7B with JumpStart (instruction-based, LoRA)
- Deploying and testing the fine-tuned model
- Cleaning up all resources

> **Prerequisite:** Accept the Mistral 7B end-user license in SageMaker Studio before running this notebook.
> Go to **Studio → JumpStart → search 'Mistral 7B' → Subscribe**.


## 1. Configuration
Edit the values in this cell to change the model, instance type, or dataset.

In [ ]:
# --- Tuneable config ---
MODEL_ID           = 'huggingface-llm-mistral-7b'
TRAINING_INSTANCE  = 'ml.g5.2xlarge'
INFERENCE_INSTANCE = 'ml.g5.2xlarge'
LOCAL_DATASET      = 'dataset.jsonl'
LOCAL_TEMPLATE     = 'template.json'
S3_KEY_PREFIX      = 'mistral-finetune-data'

# Hyperparameters
EPOCHS        = 3
LEARNING_RATE = 2e-4
BATCH_SIZE    = 4
LORA_R        = 8
LORA_ALPHA    = 16

## 2. Setup

In [ ]:
# Install the correct AWS SageMaker Python SDK (v2.x)
!pip install 'sagemaker>=2.0,<3.0' --quiet

# Restart kernel so the newly installed package is available
import sys
if 'sagemaker' in sys.modules:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Import libraries
import boto3
import sagemaker
from sagemaker.session import Session

session = Session()
role    = sagemaker.get_execution_role()
bucket  = session.default_bucket()
region  = session.boto_region_name

print(f'Region : {region}')
print(f'Bucket : {bucket}')
print(f'Role   : {role}')

## 3. Upload Dataset

The dataset uses instruction-based format with three fields:
- `instruction` — the system-level instruction
- `context` — the customer message
- `response` — the ideal agent response

A `template.json` file tells the training container how to assemble these into a prompt.

In [ ]:
# Upload dataset and template to S3
from sagemaker.s3 import S3Uploader

s3_data_path = f's3://{bucket}/{S3_KEY_PREFIX}'
S3Uploader.upload(LOCAL_DATASET, s3_data_path)
S3Uploader.upload(LOCAL_TEMPLATE, s3_data_path)
print('Uploaded training data to:', s3_data_path)

## 4. Fine-Tune with JumpStart

Mistral 7B is fine-tuned using **LoRA** (Low-Rank Adaptation) — an efficient technique that updates only a small number of parameters rather than the full model weights. This makes fine-tuning feasible on a single GPU instance.

In [ ]:
# Load JumpStart estimator for Mistral 7B
from sagemaker.jumpstart.estimator import JumpStartEstimator

estimator = JumpStartEstimator(
    model_id=MODEL_ID,
    role=role,
    instance_type=TRAINING_INSTANCE
)

In [ ]:
# Configure hyperparameters
estimator.set_hyperparameters(
    instruction_tuned='True',
    epoch=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    int8_quantization='True'
)

In [ ]:
# Start training job (blocks until complete — expect ~20-30 minutes)
estimator.fit({'train': s3_data_path}, wait=True)

## 5. Verify Training

In [ ]:
# Verify the fine-tuned model artifact and check training loss
sm_client = boto3.client('sagemaker', region_name=region)
job = sm_client.describe_training_job(
    TrainingJobName=estimator.latest_training_job.name
)
print('Training job status   :', job['TrainingJobStatus'])
print('Model artifact S3 path:', job['ModelArtifacts']['S3ModelArtifacts'])

metrics = job.get('FinalMetricDataList', [])
if metrics:
    for m in metrics:
        print(f"{m['MetricName']}: {m['Value']:.4f}")
else:
    print('No metrics available')

## 6. Deploy

In [ ]:
# Deploy the fine-tuned model
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type=INFERENCE_INSTANCE
)
print('Endpoint name:', predictor.endpoint_name)

## 7. Test

Test the fine-tuned model with customer messages it hasn't seen before.
The model should respond in the professional, empathetic tone it was trained on.

In [ ]:
# Test the fine-tuned model with unseen customer messages
test_messages = [
    'My order has not arrived and it has been three weeks.',
    'I was charged twice and I want my money back.',
    'The product I received is broken.',
]

for message in test_messages:
    prompt = (
        'Below is a customer support interaction. Respond professionally and empathetically.\n\n'
        '### Instruction:\n'
        'You are a helpful customer support agent. Respond professionally and empathetically to the customer message.\n\n'
        f'### Customer message:\n{message}\n\n'
        '### Response:\n'
    )
    response = predictor.predict({
        'inputs': prompt,
        'parameters': {
            'max_new_tokens': 150,
            'do_sample': False,
            'temperature': 0.7
        }
    })
    generated = response[0]['generated_text'].split('### Response:')[-1].strip()
    print(f'Customer: {message}')
    print(f'Agent   : {generated}')
    print()

## 8. Cleanup
**Run this cell as soon as you finish testing to avoid ongoing charges.**

In [ ]:
# Delete ALL resources to avoid ongoing charges
s3_client = boto3.client('s3', region_name=region)

# 1. Delete the endpoint (also removes endpoint config and model automatically)
try:
    predictor.delete_endpoint()
    print('✓ Endpoint deleted')
except Exception as e:
    print(f'Endpoint deletion skipped: {e}')

# 2. Delete uploaded training data from S3
try:
    for key in [LOCAL_DATASET, LOCAL_TEMPLATE]:
        s3_client.delete_object(Bucket=bucket, Key=f'{S3_KEY_PREFIX}/{key}')
    print('✓ S3 training data deleted')
except Exception as e:
    print(f'S3 data deletion skipped: {e}')

# 3. Delete training job output artifacts from S3
try:
    training_job_name = estimator.latest_training_job.name
    prefix = f'{training_job_name}/output/'
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    objects = response.get('Contents', [])
    if objects:
        s3_client.delete_objects(
            Bucket=bucket,
            Delete={'Objects': [{'Key': obj['Key']} for obj in objects]}
        )
        print(f'✓ Training artifacts deleted ({len(objects)} objects)')
    else:
        print('No training artifacts found in S3')
except Exception as e:
    print(f'Training artifact deletion skipped: {e}')

print('\nCleanup complete. Verify in the AWS Console that no endpoints or models remain.')